# Market Intelligence, Competitive Analysis & Geospatial Intelligence

**Guided Jupyter notebook · Olist Brazilian E-Commerce Public Dataset**

This notebook walks through the project's reproducible Python pipeline and explains how the prepared metrics feed the Vite dashboard. It covers data profiling, careful relational joins, category and seller benchmarking, K-Means seller cohorts, postcode geolocation enrichment, buyer/seller footprint analysis and straight-line distance proxies.

> **Scope:** the results describe the historical Olist marketplace sample (2016–2018). They do not represent all Brazilian e-commerce, national market share, current demand or road-network delivery routes.

## Before running

1. Download the [Olist Brazilian E-Commerce dataset from Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce).
2. Extract the archive and place the **nine required CSVs** in `data/olist/`, including `olist_geolocation_dataset.csv`.
3. From the repository root, install `requirements.txt` and JupyterLab.
4. Run the cells from top to bottom.

The notebook imports the same functions used by `run_analysis.py`; it is not a separate implementation of the dashboard logic.

## 1. Environment and project setup

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

# Support launching Jupyter either from the project root or notebooks/.
ROOT = Path.cwd()
if not (ROOT / "src" / "pipeline.py").exists() and (ROOT.parent / "src" / "pipeline.py").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.pipeline import (
    FILES,
    load_sources,
    prepare_geolocation_lookup,
    build_item_fact,
    build_analytics,
    build_geospatial_analytics,
    haversine_km,
    run_pipeline,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
print("Project root:", ROOT.resolve())

## 2. Load the nine source tables

In [ ]:
raw = load_sources()
source_inventory = pd.DataFrame([
    {"table": name, "rows": len(frame), "columns": len(frame.columns)}
    for name, frame in raw.items()
]).sort_values("table")
display(source_inventory)
assert "geolocation" in raw, "The postcode geolocation CSV is required for this project."

## 3. Inspect table schemas and data quality

In [ ]:
for name, frame in raw.items():
    print(f"\n{name.upper()} — {len(frame):,} rows × {len(frame.columns)} columns")
    display(pd.DataFrame({
        "column": frame.columns,
        "dtype": frame.dtypes.astype(str).values,
        "missing": frame.isna().sum().values,
        "missing_pct": (frame.isna().mean().values * 100).round(2),
    }))

## 4. Geolocation: understand the postcode lookup

The geolocation table is not one row per postcode. It contains multiple coordinate observations for a postcode prefix. Joining it directly to customer or seller tables would create a one-to-many join and could multiply transaction rows. We first deduplicate exact records, validate coordinates, and aggregate to one median centroid per five-digit prefix.

In [ ]:
geo_raw = raw["geolocation"].copy()
print("Source geolocation rows:", f"{len(geo_raw):,}")
print("Exact duplicate rows:", f"{geo_raw.duplicated().sum():,}")
print("Unique raw postcode prefixes:", f"{geo_raw['geolocation_zip_code_prefix'].nunique():,}")
display(geo_raw.head())

In [ ]:
geo_lookup, geo_audit = prepare_geolocation_lookup(geo_raw)
print("Geolocation preparation audit:")
display(pd.Series(geo_audit, name="value").to_frame())
print("One row per postcode prefix:", geo_lookup["zip_code_prefix"].is_unique)
display(geo_lookup.head())

In [ ]:
# Inspect the spatial extent of the postcode centroids.
geo_lookup[["geo_lat", "geo_lng"]].describe().T

## 5. Build the delivered-order item fact table

The analytical grain is one row per order item. Order-level reviews and payments are aggregated before joining to item-level data. Customer and seller postcode centroids are joined many-to-one after postcode aggregation.

In [ ]:
fact, orders_all, audit = build_item_fact(raw)
print(f"Prepared delivered item rows: {len(fact):,}")
print(f"Delivered orders: {fact['order_id'].nunique():,}")
print("Purchase period:", fact["order_purchase_timestamp"].min(), "to", fact["order_purchase_timestamp"].max())
display(fact.head(5))

## 6. Review the preparation audit

In [ ]:
audit_summary = {k: v for k, v in audit.items() if k not in {"payment_methods_delivered", "geo_lookup"}}
display(pd.Series(audit_summary, dtype=object).to_frame("value"))
print("Geolocation audit:")
display(pd.Series(audit["geolocation_audit"], name="value").to_frame())

## 7. Core marketplace performance

In [ ]:
gross_gmv = float(fact["item_gmv"].sum())
order_count = int(fact["order_id"].nunique())
active_sellers = int(fact["seller_id"].nunique())
active_categories = int(fact["category"].nunique())
identified_buyers = int(fact["customer_unique_id"].nunique())
avg_order_value = gross_gmv / max(order_count, 1)

core_kpis = pd.DataFrame([
    ("Delivered item GMV (BRL)", gross_gmv),
    ("Delivered orders", order_count),
    ("Active sellers", active_sellers),
    ("Active categories", active_categories),
    ("Identified buyers", identified_buyers),
    ("Average order value (BRL)", avg_order_value),
], columns=["metric", "value"])
display(core_kpis)

## 8. Monthly demand trend

In [ ]:
monthly = (fact.groupby("purchase_month", as_index=False)
           .agg(gmv=("item_gmv", "sum"), orders=("order_id", "nunique")))
display(monthly.head())
fig = px.area(monthly, x="purchase_month", y="gmv", title="Monthly delivered item GMV", labels={"purchase_month": "Purchase month", "gmv": "GMV (BRL)"})
fig.update_layout(template="plotly_white", height=380)
fig.show()

## 9. Category intelligence and seller competition

In [ ]:
category_benchmark = (fact.groupby("category", as_index=False)
    .agg(gmv=("item_gmv", "sum"), orders=("order_id", "nunique"),
         sellers=("seller_id", "nunique"), products=("product_id", "nunique"),
         average_item_price=("price", "mean")))
category_benchmark["gmv_share"] = category_benchmark["gmv"] / category_benchmark["gmv"].sum()
display(category_benchmark.sort_values("gmv", ascending=False).head(15))
fig = px.bar(category_benchmark.nlargest(15, "gmv").sort_values("gmv"),
             x="gmv", y="category", orientation="h", color="gmv_share",
             title="Top categories by delivered item GMV", color_continuous_scale="Purples")
fig.update_layout(template="plotly_white", height=560)
fig.show()

## 10. Seller benchmarking and concentration

In [ ]:
seller_benchmark = (fact.groupby("seller_id", as_index=False)
    .agg(gmv=("item_gmv", "sum"), orders=("order_id", "nunique"),
         products=("product_id", "nunique"), categories=("category", "nunique"),
         buyers=("customer_unique_id", "nunique")))
seller_benchmark["gmv_share"] = seller_benchmark["gmv"] / seller_benchmark["gmv"].sum()
seller_benchmark = seller_benchmark.sort_values("gmv", ascending=False)
top5_share = seller_benchmark.head(5)["gmv_share"].sum()
seller_hhi = seller_benchmark["gmv_share"].pow(2).sum() * 10000
print(f"Top-five seller sample share: {top5_share:.1%}")
print(f"Seller HHI within the Olist sample: {seller_hhi:,.0f}")
display(seller_benchmark.head(15))

## 11. Relative category opportunity scoring

The pipeline combines demand scale (35%), comparable year-to-date growth (30%), inverse seller concentration (20%) and review gap (15%). This is a relative investigation-priority index, not a TAM/SAM/SOM model, forecast or causal estimate.

In [ ]:
payload = build_analytics(fact, orders_all, audit)
opportunities = pd.DataFrame(payload["opportunities"])
display(opportunities[["category", "gmv", "sellers", "yoy_growth", "hhi", "average_review", "opportunity_score"]].head(20))
fig = px.bar(opportunities.head(12).sort_values("opportunity_score"),
             x="opportunity_score", y="category", orientation="h",
             title="Relative category opportunity score", color="opportunity_score",
             color_continuous_scale="Purples")
fig.update_layout(template="plotly_white", height=500)
fig.show()

## 12. Seller cohorts with K-Means

In [ ]:
seller_cohorts = pd.DataFrame(payload["seller_cohorts"])
display(seller_cohorts)
fig = px.bar(seller_cohorts, x="seller_cohort", y="sellers", color="total_gmv",
             title="Exploratory seller cohorts", color_continuous_scale="Purples")
fig.update_layout(template="plotly_white", height=350)
fig.show()

## 13. Geospatial coverage: buyer and seller locations

Coordinates are not exact customer addresses. Each five-digit postcode prefix is represented by the median latitude/longitude of the source points mapped to that prefix. City positions are therefore approximate postcode-centroid summaries.

In [ ]:
geo = payload["geospatial"]
geo_summary = pd.Series(geo["summary"], name="value").to_frame()
display(geo_summary)
geo_points = pd.DataFrame(geo["map_points"])
print("Mapped buyer/seller city points:", len(geo_points))
display(geo_points.head())

In [ ]:
if not geo_points.empty:
    fig = px.scatter_geo(
        geo_points,
        lat="lat", lon="lon",
        color="role",
        size="intensity",
        size_max=18,
        hover_name="city_label",
        hover_data={"state": True, "gmv": ":,.0f", "orders": ":,", "entities": ":,", "lat": ":.3f", "lon": ":.3f"},
        scope="south america",
        projection="mercator",
        title="Buyer demand and seller supply footprint — postcode centroids",
        labels={"role": "Location role", "intensity": "Activity"},
    )
    fig.update_geos(showland=True, landcolor="#f4f5f9", showcountries=True,
                    countrycolor="#cfd4df", showcoastlines=True, coastlinecolor="#cfd4df",
                    showocean=True, oceancolor="#f9fbff")
    fig.update_layout(template="plotly_white", height=620, margin=dict(l=10,r=10,t=55,b=10))
    fig.show()
else:
    print("No mapped points. Confirm that the geolocation CSV and postcode columns are present.")

## 14. Regional buyer demand versus seller presence

For each state, seller-presence gap is calculated as **buyer share minus active-seller share**. Positive values show that the state represents a larger share of identified buyers than active sellers within this dataset. It is a descriptive footprint comparison—not proof of a service gap or unmet demand.

In [ ]:
state_balance = pd.DataFrame(geo["state_balance"])
display(state_balance.sort_values("seller_presence_gap_pp", ascending=False).head(15))
fig = px.bar(
    state_balance.query("state != 'Unknown'").nlargest(15, "seller_presence_gap_pp").sort_values("seller_presence_gap_pp"),
    x="seller_presence_gap_pp", y="state", orientation="h",
    title="States where buyer share exceeds seller presence share",
    labels={"seller_presence_gap_pp": "Buyer share − seller share (percentage points)", "state": "State"},
    color="seller_presence_gap_pp", color_continuous_scale="Purples",
)
fig.update_layout(template="plotly_white", height=500)
fig.show()

## 15. Seller-to-buyer geographic separation

For each unique order–seller pair, the project calculates a Haversine great-circle distance between postcode centroids. This is a **straight-line distance proxy**, not a road distance, carrier route or delivery-time estimate.

In [ ]:
distance_pairs = pd.DataFrame(geo["distance_pairs"])
distance_buckets = pd.DataFrame(geo["distance_buckets"])
display(distance_buckets)
display(distance_pairs.head(15))

In [ ]:
if not distance_buckets.empty:
    fig = px.bar(distance_buckets, x="distance_band_km", y="order_seller_pairs",
                 title="Order–seller pairs by straight-line distance band",
                 labels={"distance_band_km": "Distance band (km)", "order_seller_pairs": "Order–seller pairs"},
                 color="order_seller_pairs", color_continuous_scale="Purples")
    fig.update_layout(template="plotly_white", height=380)
    fig.show()

## 16. Understand and sanity-check the Haversine calculation

In [ ]:
# Approximate straight-line distance between postcode-centroid examples in São Paulo and Rio de Janeiro.
example_distance = float(haversine_km([-23.55], [-46.63], [-22.90], [-43.20])[0])
print(f"Illustrative centroid-to-centroid great-circle distance: {example_distance:,.1f} km")
print("The actual pipeline applies the same calculation to mapped order–seller pairs.")

## 17. Build and write the exact dashboard data product

The next cell reruns the full pipeline from the source CSVs. It writes the aggregated JSON used by the Vite dashboard and local CSV extracts for deeper analysis. The frontend JSON should be committed to GitHub; local row-level outputs can remain excluded.

In [ ]:
final_payload = run_pipeline()
print("Dashboard status:", final_payload["status"])
print("Dashboard JSON:", ROOT / "frontend/public/data/market_intelligence.json")
print("Geospatial summary:")
display(pd.Series(final_payload["geospatial"]["summary"], name="value").to_frame())

## 18. Interpretation checklist and limitations

Before writing a portfolio conclusion, answer these questions:

1. Which categories have high observed demand and relatively fragmented seller competition?
2. Which categories have an opportunity score that is worth investigating, and which score inputs drive it?
3. How concentrated is seller GMV within the Olist sample overall and by category?
4. Which states have a higher share of identified buyers than active sellers?
5. How much of the customer/seller activity can be mapped to postcode centroids?
6. What does the median straight-line buyer–seller distance say—and what does it **not** say?
7. How sensitive are the findings to the opportunity-score weights and the historical period?

**Do not claim:** national Brazilian e-commerce market share, external TAM/SAM/SOM, profitability (the dataset does not contain full seller costs), customer addresses, road-network routes, delivery route distances, causal reasons for demand, or a prediction from the K-Means clusters.

**Dataset attribution:** Olist. *Brazilian E-Commerce Public Dataset by Olist*. Kaggle. https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce. Licence identified by Kaggle: CC BY-NC-SA 4.0. This is a non-commercial portfolio analysis; review the source licence before redistribution or any commercial use.

## 15. Aggregate views used by the dashboard

The dashboard adds three deterministic views derived from the same delivered-order records: monthly category composition (largest eight categories, grouped tail), delivery-duration distribution and the observed delivery-time / review relationship. They do not change any KPI definition or add new source data.


In [ ]:
derived = {
    "monthly_category": pd.DataFrame(payload["monthly_category"]),
    "delivery_distribution": pd.DataFrame(payload["delivery_distribution"]),
    "delivery_review": pd.DataFrame(payload["delivery_review"]),
}
display(derived["monthly_category"].head(12))
assert round(derived["monthly_category"]["gmv"].sum(), 2) == round(float(fact["item_gmv"].sum()), 2)

fig = px.area(
    derived["monthly_category"].sort_values("purchase_month"),
    x="purchase_month", y="gmv", color="category",
    title="Monthly delivered GMV by leading category (tail grouped)",
)
fig.update_layout(template="plotly_dark", height=420)
fig.show()

display(derived["delivery_distribution"])
fig = px.bar(
    derived["delivery_review"], x="delivery_band_days", y="avg_review",
    title="Average order review by delivery-duration band",
)
fig.update_layout(template="plotly_dark", height=360)
fig.show()
